# Exercise 06 — RAG Pipeline from Scratch

**RAG = Retrieval Augmented Generation** — a technique to query large documents without sending everything to Claude.

**The problem:** A 500-page document exceeds Claude's context window, and even if it fits, sending the full document every time is slow and expensive.

**The solution:** Pre-process the document once into searchable chunks, then at query time retrieve only the 2–5 most relevant chunks and include only those in the prompt.

**What this notebook builds:**
1. Text chunking (three strategies)
2. Embeddings (numeric vectors representing text meaning)
3. Vector store with cosine similarity search
4. BM25 lexical search
5. Hybrid search with Reciprocal Rank Fusion
6. LLM-based reranking
7. Contextual retrieval (add context to chunks before indexing)

In [ ]:
# Install extra dependencies if needed
# !pip install voyageai numpy

import os
import json
import math
import re
from collections import defaultdict
from dotenv import load_dotenv
import anthropic

load_dotenv(dotenv_path="../.env")
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

print("Ready!")

## Sample Document

We'll use a mock company annual report throughout this notebook.

In [ ]:
DOCUMENT = """
## Executive Summary

Acme Corp had a strong 2023 fiscal year, achieving record revenue of $450M, up 23% from 2022.
Our three business units — Software, Hardware, and Professional Services — all posted growth.
Headcount grew to 3,200 employees across 12 countries.

## Software Division

The Software Division generated $280M in revenue, representing 62% of total company revenue.
Key products: CloudDeploy (enterprise SaaS), DataPipeline (ETL automation), SecureVault (encryption).
The engineering team resolved 847 production incidents in 2023, down from 1,203 in 2022.
The software engineering team shipped 4 major product releases and 34 minor updates.
Incident 2023-0042 in Q3 caused 4 hours of downtime for CloudDeploy; root cause was a misconfigured load balancer.

## Hardware Division

Hardware revenue reached $95M driven by the launch of the EdgeNode 3000 series.
Supply chain challenges caused a 6-week delay in the EdgeNode 3000 launch.
Manufacturing partnerships with suppliers in Vietnam and Indonesia were expanded.

## Professional Services

Professional Services grew to $75M, up 40% YoY.
The team completed 312 client implementations and maintained a 94% client satisfaction score.
Three new service packages were introduced: GoldCare, PlatinumCare, and EliteCare.

## Cybersecurity & Compliance

The cybersecurity team handled 23 security incidents in 2023, none of which resulted in data breaches.
ISO 27001 certification was renewed in March 2023.
A new zero-trust network architecture was implemented across all data centers.
Penetration testing was conducted twice, in Q1 and Q3, with all critical findings remediated within 30 days.

## Human Resources

Employee headcount grew from 2,800 to 3,200 (+400 net new hires).
Voluntary attrition rate was 8.2%, below the industry average of 13%.
A new remote-first policy was adopted, allowing 80% of roles to be fully remote.
Learning & Development budget increased by 35% to $4.2M.

## Financial Highlights

Total revenue: $450M (+23% YoY)
Gross margin: 68%
EBITDA: $112M (25% margin)
Cash and equivalents: $89M
R&D spend: $67M (15% of revenue)
"""

print(f"Document length: {len(DOCUMENT)} characters")

## Part 1 — Text Chunking

**Three strategies:**

1. **Size-based** — split every N characters, with overlap to preserve context across boundaries
2. **Structure-based** — split on document structure (headers, paragraphs)
3. **Semantic** — group consecutive sentences that are semantically similar (advanced)

For most documents, structure-based chunking gives the best results.

In [ ]:
def chunk_by_size(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """Split text into equal character chunks with overlap to preserve context."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap  # overlap with previous chunk
    return [c.strip() for c in chunks if c.strip()]


def chunk_by_section(text: str) -> list[str]:
    """Split markdown text on ## headers so each section is one chunk."""
    sections = re.split(r'(?=^## )', text, flags=re.MULTILINE)
    return [s.strip() for s in sections if s.strip()]


# Compare strategies
size_chunks     = chunk_by_size(DOCUMENT, chunk_size=400, overlap=80)
section_chunks  = chunk_by_section(DOCUMENT)

print(f"Size-based chunks:    {len(size_chunks)}")
print(f"Section-based chunks: {len(section_chunks)}")
print()
print("First section chunk:")
print(section_chunks[0][:400])

## Part 2 — Embeddings & Vector Store

An **embedding** is a list of numbers (e.g. 1024 floats) that represents the meaning of a piece of text. Similar texts produce similar vectors. We measure similarity using **cosine similarity**.

For simplicity in this notebook, we'll simulate embeddings using a lightweight bag-of-words approach. In production, use a real embedding model (Voyage AI, OpenAI, etc.).

In [ ]:
import hashlib


def simple_embed(text: str, dims: int = 64) -> list[float]:
    """
    Lightweight deterministic fake embedding for demo purposes.
    In production, replace with: voyageai.Client().embed([text], model='voyage-3').embeddings[0]
    """
    words = re.findall(r'\w+', text.lower())
    vec = [0.0] * dims
    for word in words:
        h = int(hashlib.md5(word.encode()).hexdigest(), 16)
        for i in range(min(4, dims)):
            idx = (h >> (i * 8)) % dims
            vec[idx] += 1.0
    # L2 normalise
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors. Result range: -1 to 1."""
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x * x for x in a))
    mag_b = math.sqrt(sum(x * x for x in b))
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot / (mag_a * mag_b)


class VectorStore:
    """Simple in-memory vector database."""

    def __init__(self):
        self._vectors: list[tuple[list[float], dict]] = []

    def add(self, embedding: list[float], metadata: dict) -> None:
        self._vectors.append((embedding, metadata))

    def search(self, query_embedding: list[float], top_k: int = 3) -> list[dict]:
        """Return top_k most similar documents sorted by cosine similarity."""
        scored = [
            {"score": cosine_similarity(query_embedding, vec), **meta}
            for vec, meta in self._vectors
        ]
        return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]


# Build the vector store
chunks = chunk_by_section(DOCUMENT)
store  = VectorStore()

for i, chunk in enumerate(chunks):
    embedding = simple_embed(chunk)
    store.add(embedding, {"id": i, "content": chunk})

print(f"Indexed {len(chunks)} chunks into vector store.")

# Test a query
query  = "What incidents did the engineering team resolve?"
q_emb  = simple_embed(query)
results = store.search(q_emb, top_k=2)

print(f"\nQuery: {query!r}")
for r in results:
    print(f"  score={r['score']:.3f} | {r['content'][:100]}...")

## Part 3 — BM25 Lexical Search

**BM25 (Best Match 25)** is a classic keyword-based search algorithm. Unlike embeddings it:
- Scores documents by how often query terms appear in them
- Penalises very common words (like 
, 
) — rare terms matter more
- Is great at finding exact term matches that semantic search can miss

The hybrid approach uses **both** vector search and BM25, then merges results.

In [ ]:
class BM25Index:
    """Minimal BM25 implementation for keyword search."""

    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b
        self._docs: list[dict]          = []
        self._tokenized: list[list[str]]= []
        self._df: dict[str, int]        = defaultdict(int)  # document frequency

    def _tokenize(self, text: str) -> list[str]:
        return re.findall(r'\w+', text.lower())

    def add(self, metadata: dict) -> None:
        tokens = self._tokenize(metadata["content"])
        self._docs.append(metadata)
        self._tokenized.append(tokens)
        for token in set(tokens):
            self._df[token] += 1

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        query_terms = self._tokenize(query)
        N   = len(self._docs)
        avg_dl = sum(len(t) for t in self._tokenized) / (N or 1)

        scores = []
        for idx, (doc, tokens) in enumerate(zip(self._docs, self._tokenized)):
            tf_map = defaultdict(int)
            for t in tokens:
                tf_map[t] += 1
            dl = len(tokens)
            score = 0.0
            for term in query_terms:
                tf  = tf_map.get(term, 0)
                df  = self._df.get(term, 0)
                idf = math.log((N - df + 0.5) / (df + 0.5) + 1)
                tf_norm = (tf * (self.k1 + 1)) / (
                    tf + self.k1 * (1 - self.b + self.b * dl / avg_dl)
                )
                score += idf * tf_norm
            scores.append({"score": score, **doc})

        return sorted(scores, key=lambda x: x["score"], reverse=True)[:top_k]


# Build the BM25 index
bm25 = BM25Index()
for i, chunk in enumerate(chunks):
    bm25.add({"id": i, "content": chunk})

print("BM25 index built.")

# Compare results for a specific query
query = "What incidents did the engineering team resolve?"
bm25_results = bm25.search(query, top_k=2)
print(f"\nBM25 results for: {query!r}")
for r in bm25_results:
    print(f"  score={r['score']:.3f} | {r['content'][:100]}...")

## Part 4 — Hybrid Search with Reciprocal Rank Fusion

**Reciprocal Rank Fusion (RRF)** combines ranked lists from different search systems:

$$\text{RRF\_score}(d) = \sum_{\text{method}} \frac{1}{\text{rank}(d) + 1}$$

A document that ranks #1 in both vector AND BM25 search gets the highest combined score.

In [ ]:
def reciprocal_rank_fusion(ranked_lists: list[list[dict]], top_k: int = 3) -> list[dict]:
    """
    Merge multiple search result lists using RRF.
    Each result dict must have an 'id' field.
    """
    rrf_scores: dict[int, float] = defaultdict(float)
    doc_lookup: dict[int, dict]  = {}

    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list):
            doc_id = doc["id"]
            rrf_scores[doc_id] += 1.0 / (rank + 1)
            doc_lookup[doc_id] = doc

    sorted_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return [
        {"rrf_score": rrf_scores[doc_id], **doc_lookup[doc_id]}
        for doc_id in sorted_ids[:top_k]
    ]


class HybridRetriever:
    """Combines vector search and BM25 with RRF."""

    def __init__(self, chunks: list[str]):
        self.chunks = chunks
        self.vector_store = VectorStore()
        self.bm25_index   = BM25Index()
        for i, chunk in enumerate(chunks):
            meta = {"id": i, "content": chunk}
            self.vector_store.add(simple_embed(chunk), meta)
            self.bm25_index.add(meta)

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        vec_results  = self.vector_store.search(simple_embed(query), top_k=top_k * 2)
        bm25_results = self.bm25_index.search(query, top_k=top_k * 2)
        return reciprocal_rank_fusion([vec_results, bm25_results], top_k=top_k)


retriever = HybridRetriever(chunks)

# Test queries
test_queries = [
    "What did the software engineering team accomplish?",
    "Tell me about the security incidents in 2023",
    "What was the total revenue?",
]

for q in test_queries:
    results = retriever.search(q, top_k=2)
    print(f"Query: {q!r}")
    for r in results:
        header = r['content'].split('\n')[0]
        print(f"  rrf={r['rrf_score']:.3f} | {header}")
    print()

## Part 5 — LLM Reranking

After hybrid search we can pass the top candidates to Claude and ask it to reorder them by relevance. This catches cases where the query phrasing doesn't exactly match but the meaning does (e.g. "ENG team" vs "software engineering team").

In [ ]:
RERANK_PROMPT = """You are a relevance judge. Given a user query and a list of document chunks,
return the IDs of the most relevant chunks in decreasing order of relevance.
Only include chunks that are genuinely useful for answering the query.

User query: {query}

Candidate documents:
{candidates}

Return a JSON array of document IDs in order from most to least relevant.
Example: [2, 0, 1]"""


def rerank(query: str, candidates: list[dict]) -> list[dict]:
    """Use Claude to reorder search results by relevance."""
    if not candidates:
        return []

    # Pass doc IDs + first 200 chars of content to Claude
    candidates_text = "\n".join(
        f"[ID {c['id']}] {c['content'][:200]}" for c in candidates
    )

    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        stop_sequences=["]"],  # stop after the closing bracket
        messages=[
            {
                "role": "user",
                "content": RERANK_PROMPT.format(query=query, candidates=candidates_text)
            },
            {"role": "assistant", "content": "["}  # pre-fill opening bracket
        ]
    )

    try:
        ranked_ids = json.loads("[" + response.content[0].text.strip())
    except json.JSONDecodeError:
        return candidates  # fallback to original order if parsing fails

    id_to_doc = {c["id"]: c for c in candidates}
    return [id_to_doc[i] for i in ranked_ids if i in id_to_doc]


# Test reranking
query = "What did the ENG team do with Incident 2023?"
initial_results = retriever.search(query, top_k=4)
reranked_results = rerank(query, initial_results)

print(f"Query: {query!r}")
print("\nBefore reranking:")
for r in initial_results:
    print(f"  [{r['id']}] {r['content'].split(chr(10))[0][:60]}")

print("\nAfter reranking:")
for r in reranked_results:
    print(f"  [{r['id']}] {r['content'].split(chr(10))[0][:60]}")

## Part 6 — Contextual Retrieval

When a chunk is extracted from a large document, it can lose context. For example, a chunk that says _"It was resolved within 30 days"_ doesn't mention what "it" refers to.

**Contextual retrieval** pre-processes each chunk by asking Claude to generate a short context sentence that explains how the chunk relates to the broader document.

In [ ]:
ADD_CONTEXT_PROMPT = """Here is a document:
<document>
{source_document}
</document>

Here is the specific chunk from that document:
<chunk>
{chunk}
</chunk>

Write a single sentence that situates this chunk within the broader document context.
Explain what topic or section of the document this chunk is from, and any important
cross-references. Be concise — one sentence only."""


def add_context(chunk: str, source_document: str) -> str:
    """Prepend a Claude-generated context sentence to the chunk."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=150,
        messages=[{
            "role": "user",
            "content": ADD_CONTEXT_PROMPT.format(
                source_document=source_document[:3000],  # truncate if too long
                chunk=chunk
            )
        }]
    )
    context_sentence = response.content[0].text.strip()
    return f"{context_sentence}\n\n{chunk}"


# Demonstrate on one chunk
sample_chunk = chunks[1]  # Software Division section
contextualized = add_context(sample_chunk, DOCUMENT)

print("Original chunk (first 200 chars):")
print(sample_chunk[:200])
print()
print("Contextualized chunk (first 300 chars):")
print(contextualized[:300])

In [ ]:
# Build a contextualized retriever (adds context to every chunk — runs 1 API call per chunk)
print("Building contextualized index (may take ~30s)...")
contextualized_chunks = [add_context(c, DOCUMENT) for c in chunks]
contextualized_retriever = HybridRetriever(contextualized_chunks)

print(f"Indexed {len(contextualized_chunks)} contextualized chunks.")

## Part 7 — The Full RAG Pipeline

Put it all together: retrieve relevant chunks + pass them to Claude as context.

In [ ]:
RAG_PROMPT = """You are a helpful assistant. Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information."

<context>
{context}
</context>

Question: {question}"""


def rag_query(question: str, retriever: HybridRetriever, top_k: int = 3, use_rerank: bool = True) -> str:
    """Full RAG pipeline: retrieve → rerank → answer."""
    candidates = retriever.search(question, top_k=top_k * 2)
    if use_rerank:
        candidates = rerank(question, candidates)

    context = "\n\n---\n\n".join(r["content"] for r in candidates[:top_k])

    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{
            "role": "user",
            "content": RAG_PROMPT.format(context=context, question=question)
        }]
    )
    return response.content[0].text


# Test queries
questions = [
    "How many incidents did the engineering team handle in 2023?",
    "What was the company's total revenue in 2023?",
    "What happened during Incident 2023-0042?",
    "What is the employee attrition rate?",
]

for q in questions:
    print(f"Q: {q}")
    answer = rag_query(q, contextualized_retriever)
    print(f"A: {answer}")
    print()

## Summary

You built a complete RAG pipeline from scratch:

| Component | What it does |
|-----------|-------------|
| `chunk_by_section()` | Splits document into topic-based pieces |
| `simple_embed()` / Voyage AI | Converts text to a vector for semantic comparison |
| `VectorStore` | Finds semantically similar chunks |
| `BM25Index` | Finds keyword-matching chunks |
| `HybridRetriever` | Combines both with Reciprocal Rank Fusion |
| `rerank()` | Uses Claude to reorder by semantic relevance |
| `add_context()` | Enriches chunks with document context before indexing |
| `rag_query()` | Full pipeline: retrieve → rerank → answer |

**Next:** [07_advanced_features.ipynb](07_advanced_features.ipynb) — extended thinking, images, PDFs, citations, and prompt caching